# ERIP — Build Report Context

Creates the one-row Gold table `gold_report_context` used by Power BI header cards.

**Outputs**
- Portfolio **as-of date**: maximum reporting/snapshot date in the exposure fact.
- **Refresh timestamp**: UTC time at which this notebook creates the context row.
- Reusable report-view labels and reporting currency.

Attach the ERIP Lakehouse (`lh_erip`) before running this notebook. Run it after the Gold exposure fact has been built.

In [1]:
# Configuration — normally no change is required.
# Set either value explicitly only if automatic resolution reports an ambiguity.
SOURCE_TABLE = None
AS_OF_DATE_COLUMN = "gold_updated_timestamp"

OUTPUT_TABLE = "gold_report_context"
REPORT_VIEW_NAME = "CRO View"
REPORT_VIEW_ROLE = "Chief Risk Officer"
REPORTING_CURRENCY = "EUR"

SOURCE_TABLE_CANDIDATES = [
    "gold_fact_loan_exposure",
    "fact_loan_exposure",
    "gold_credit_exposure",
]

AS_OF_COLUMN_CANDIDATES = [
    "reporting_date",
    "as_of_date",
    "snapshot_date",
    "exposure_date",
    "report_date",
]

StatementMeta(, 02bd44f0-9e54-4a63-8a28-a5facb65560e, 3, Finished, Available, Finished, False)

In [2]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

available_tables = [table.name for table in spark.catalog.listTables()]
table_lookup = {name.lower(): name for name in available_tables}

if SOURCE_TABLE is None:
    matched_tables = [
        table_lookup[name.lower()]
        for name in SOURCE_TABLE_CANDIDATES
        if name.lower() in table_lookup
    ]

    if len(matched_tables) != 1:
        exposure_tables = [
            name for name in available_tables
            if "exposure" in name.lower() or "loan" in name.lower()
        ]
        raise ValueError(
            "Could not resolve exactly one exposure source table. "
            f"Candidate matches: {matched_tables}. "
            f"Available exposure/loan tables: {exposure_tables}. "
            "Set SOURCE_TABLE explicitly in the configuration cell."
        )

    SOURCE_TABLE = matched_tables[0]

source_df = spark.table(SOURCE_TABLE)
column_lookup = {column.lower(): column for column in source_df.columns}

if AS_OF_DATE_COLUMN is None:
    matched_columns = [
        column_lookup[name.lower()]
        for name in AS_OF_COLUMN_CANDIDATES
        if name.lower() in column_lookup
    ]

    if len(matched_columns) != 1:
        date_like_columns = [
            name for name in source_df.columns
            if "date" in name.lower() or "time" in name.lower()
        ]
        raise ValueError(
            "Could not resolve exactly one as-of date column. "
            f"Candidate matches: {matched_columns}. "
            f"Available date/time columns: {date_like_columns}. "
            "Set AS_OF_DATE_COLUMN explicitly in the configuration cell."
        )

    AS_OF_DATE_COLUMN = matched_columns[0]

print(f"Exposure source table: {SOURCE_TABLE}")
print(f"As-of date column: {AS_OF_DATE_COLUMN}")
print(f"Output table: {OUTPUT_TABLE}")

StatementMeta(, 02bd44f0-9e54-4a63-8a28-a5facb65560e, 4, Finished, Available, Finished, False)

Exposure source table: fact_loan_exposure
As-of date column: gold_updated_timestamp
Output table: gold_report_context


In [3]:
portfolio_as_of_date = (
    source_df
        .agg(
            F.max(F.to_date(F.col(AS_OF_DATE_COLUMN))).alias("as_of_date")
        )
        .first()["as_of_date"]
)

if portfolio_as_of_date is None:
    raise ValueError(
        f"No valid date was found in {SOURCE_TABLE}.{AS_OF_DATE_COLUMN}."
    )

refreshed_at_utc = datetime.now(timezone.utc)

report_context = spark.createDataFrame(
    [(
        1,
        portfolio_as_of_date,
        refreshed_at_utc,
        REPORT_VIEW_NAME,
        REPORT_VIEW_ROLE,
        REPORTING_CURRENCY,
        SOURCE_TABLE,
    )],
    """
    report_context_id integer,
    as_of_date date,
    refreshed_at_utc timestamp,
    view_name string,
    view_role string,
    reporting_currency string,
    source_table string
    """,
)

(
    report_context.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(OUTPUT_TABLE)
)

StatementMeta(, 02bd44f0-9e54-4a63-8a28-a5facb65560e, 5, Finished, Available, Finished, False)

In [4]:
# Quality gates and final verification
saved_context = spark.table(OUTPUT_TABLE)
saved_rows = saved_context.count()
null_required_values = (
    saved_context
        .filter(
            F.col("as_of_date").isNull()
            | F.col("refreshed_at_utc").isNull()
            | F.col("view_name").isNull()
            | F.col("view_role").isNull()
        )
        .count()
)

assert saved_rows == 1, f"Expected one context row; found {saved_rows}."
assert null_required_values == 0, "Required report-context values contain nulls."
assert portfolio_as_of_date <= refreshed_at_utc.date(), (
    "The portfolio as-of date cannot be later than the refresh date."
)

print("gold_report_context created successfully.")
display(saved_context)

StatementMeta(, 02bd44f0-9e54-4a63-8a28-a5facb65560e, 6, Finished, Available, Finished, False)

gold_report_context created successfully.


SynapseWidget(Synapse.DataFrame, 4ca430b4-4c47-4f9e-8ea7-65038a4cd517)